# EDA Superstore - ComprasOnline.com

Cuaderno de trabajo para documentar el análisis exploratorio del obligatorio.

El objetivo no es solo calcular métricas, sino encontrar hallazgos que puedan transformarse en visualizaciones claras para el informe y la presentación.

## Setup

Carga de librerías y lectura del archivo Excel original.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

DATA_PATH = Path('..') / 'data' / 'raw' / 'Sample - Superstore v1.0.xlsx'
CHARTS_DIR = Path('..') / 'outputs' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name='Sample - Superstore')
df['Year'] = df['Order Date'].dt.year

df.head()

In [ ]:
df.info()

## Pregunta 2 - Análisis por Estado

**Pregunta del cliente:** ¿Cómo ha sido la venta por Estado? Ver evolución y observar especialmente el último año.

**Quiero mostrar que:** las ventas están concentradas en pocos estados y que, al mirar 2017, algunos estados con ventas altas no necesariamente fueron rentables.

**Campos utilizados:**

- `State`: estado.
- `Order Date`: fecha de la orden, usada para extraer el año.
- `Sales`: ventas, métrica principal del eje X.
- `Profit`: ganancia/pérdida, usada como lectura complementaria y color.

### 2.1 Ranking acumulado de ventas por Estado

Primero miramos qué estados explican mayor volumen de ventas en todo el período 2014-2017.

In [ ]:
state_total = (
    df.groupby('State')
    .agg(
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Order ID', 'nunique')
    )
    .sort_values('Sales', ascending=False)
)

state_total.head(12).round(2)

In [ ]:
top5_share = state_total.head(5)['Sales'].sum() / state_total['Sales'].sum()

print(f"Los 5 estados con más ventas concentran {top5_share:.1%} de las ventas totales.")

### 2.2 Evolución anual de los principales estados

Tomamos los 12 estados con más ventas acumuladas y observamos cómo evolucionaron año a año.

In [ ]:
top_states = state_total.head(12).index

state_year_sales = (
    df[df['State'].isin(top_states)]
    .groupby(['State', 'Year'])['Sales']
    .sum()
    .unstack(fill_value=0)
    .loc[top_states]
)

state_year_sales.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

state_year_sales.plot(
    kind='barh',
    ax=ax,
    color=['#9aa6b2', '#7b8794', '#4f6f91', '#1f4e79'],
    width=0.78
)

ax.invert_yaxis()
ax.set_title('California lidera, pero Washington acelera con fuerza en 2017', loc='left', fontsize=15, fontweight='bold')
ax.set_xlabel('Ventas')
ax.set_ylabel('Estado')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f'${x/1000:.0f}k'))
ax.grid(axis='x', alpha=0.25)
ax.legend(title='Año', frameon=False, ncol=4, loc='lower right')
ax.spines[['top', 'right', 'left']].set_visible(False)

fig.tight_layout()
fig.savefig(CHARTS_DIR / 'estado_ventas_evolucion_top12.png', dpi=180)
plt.show()

### 2.3 Foco en el último año disponible: 2017

Para 2017, usamos `Sales` como largo de la barra y `Profit` como señal de rentabilidad. Esto permite detectar estados que venden mucho, pero generan pérdida.

In [ ]:
state_2017 = (
    df[df['Year'] == 2017]
    .groupby('State')
    .agg(
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Order ID', 'nunique')
    )
    .sort_values('Sales', ascending=False)
)

state_2017.head(12).round(2)

In [ ]:
top_2017 = state_2017.head(12).copy()
loss_states_2017 = top_2017[top_2017['Profit'] < 0]

print('Estados del top 12 de ventas 2017 con pérdida:')
print(', '.join(loss_states_2017.index))
print(f"Pérdida conjunta: ${loss_states_2017['Profit'].sum():,.2f}")

In [ ]:
colors = ['#2e7d32' if profit >= 0 else '#c62828' for profit in top_2017['Profit']]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top_2017.index, top_2017['Sales'], color=colors)

ax.invert_yaxis()
ax.set_title('En 2017, ventas altas no siempre significaron rentabilidad', loc='left', fontsize=15, fontweight='bold')
ax.set_xlabel('Ventas 2017')
ax.set_ylabel('Estado')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f'${x/1000:.0f}k'))
ax.grid(axis='x', alpha=0.25)
ax.spines[['top', 'right', 'left']].set_visible(False)

max_sales = top_2017['Sales'].max()
for bar, profit in zip(bars, top_2017['Profit']):
    ax.text(
        bar.get_width() + max_sales * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'Ganancia: ${profit/1000:.1f}k',
        va='center',
        fontsize=9,
        color='#2b2b2b'
    )

ax.text(0.98, 0.03, 'Verde: ganancia positiva | Rojo: pérdida', transform=ax.transAxes, ha='right', fontsize=9, color='#555')

fig.tight_layout()
fig.savefig(CHARTS_DIR / 'estado_ventas_2017_rentabilidad.png', dpi=180)
plt.show()

### 2.4 Lectura para el informe

**Hallazgo:** California y New York concentran grandes ventas y mantienen ganancias positivas. Washington crece con fuerza en 2017. Sin embargo, Texas, Pennsylvania, Illinois, North Carolina y Ohio aparecen entre los estados con más ventas de 2017 y aun así tienen pérdidas.

**Interpretación:** no conviene evaluar los estados solo por volumen de ventas. Para decidir dónde invertir, ComprasOnline.com debería distinguir entre mercados grandes y rentables, como California, New York y Washington, y mercados grandes pero problemáticos, como Texas o Pennsylvania.

**Siguiente análisis sugerido:** revisar descuentos, categorías y subcategorías en los estados con pérdida para entender qué puede estar deteriorando la rentabilidad.